# **II. Análisis inicial: Exploración de las variables en los datos oncológicos del GRD**

## Objetivo
Este notebook realiza un análisis exploratorio inicial de los datos oncológicos filtrados en la etapa anterior. Se busca:
- Identificar la estructura y las columnas disponibles en cada dataset por año
- Detectar si hay inconsistencias en los datasets (entre los años)
- Entender qué cambios se han realizado en la recolección/codificación de datos del GRD

## Proceso
1. **Lectura y exploración**: Carga los 6 archivos de datos oncológicos (2019-2024) y examina su estructura
2. **Análisis de columnas**: Identifica qué columnas contiene cada archivo y cuántos registros hay
3. **Comparación de variables**: Detecta diferencias en la estructura de datos entre años consecutivos
4. **Identificación de cambios**: Distingue entre cambios de codificación de caracteres y cambios reales en variables

### **1. Exploración de Columnas del GRD**

- El sistema de Grupos Relacionados por el Diagnóstico (GRD) contiene múltiples variables sobre episodios hospitalarios de pacientes oncológicos.
- Cada archivo debe contener las mismas columnas para permitir análisis comparativos entre años.
- Este análisis mostrará el total de columnas y filas en cada año, permitiendo identificar si hay cambios estructurales en los datasets de cada año.

In [ ]:
import pandas as pd # Para manipulación de datos
import os # Para manejo de rutas y archivos
import warnings # Para suprimir advertencias durante la carga de datos
# Suprimir advertencias sobre cambios de tipo de dato (dtype)
warnings.simplefilter(action='ignore', category=pd.errors.DtypeWarning)

# ==================== DEFINICIÓN DE RUTAS Y ARCHIVOS ====================
# Ruta donde se encuentran los archivos oncológicos filtrados
carpeta = "../../Datos/Datos oncológicos (sin procesar)"

# Lista de archivos a analizar, uno por cada año disponible
archivos = [
    "GRD_ONCOLOGIA_2019.csv",
    "GRD_ONCOLOGIA_2020.csv",
    "GRD_ONCOLOGIA_2021.csv",
    "GRD_ONCOLOGIA_2022.csv",
    "GRD_ONCOLOGIA_2023.csv",
    "GRD_ONCOLOGIA_2024.csv"
]

# ==================== EXPLORACIÓN DE ESTRUCTURA ====================
# Iterar sobre cada archivo y mostrar información sobre su estructura
for archivo in archivos:
    ruta = os.path.join(carpeta, archivo)
    # Cargar todo el dataset sin limitar filas para un análisis completo
    df = pd.read_csv(ruta, low_memory=False)
    # Extraer información sobre estructura del dataset
    columnas = df.columns.tolist()
    cantidad_columnas = len(columnas)
    cantidad_filas = len(df)
    # Mostrar resumen del archivo
    print(f"Archivo: {archivo}")
    print(f"Columnas: {columnas}")
    print(f"Cantidad de columnas: {cantidad_columnas}")
    print(f"Cantidad de filas: {cantidad_filas}")
    print("-" * 50)

Archivo: GRD_ONCOLOGIA_2019.csv
Columnas: ['COD_HOSPITAL', 'CIP_ENCRIPTADO', 'SEXO', 'FECHA_NACIMIENTO', 'ETNIA', 'PROVINCIA', 'COMUNA', 'NACIONALIDAD', 'PREVISION', 'SERVICIO_SALUD', 'TIPO_PROCEDENCIA', 'TIPO_INGRESO', 'ESPECIALIDAD_MEDICA', 'TIPO_ACTIVIDAD', 'FECHA_INGRESO', 'SERVICIOINGRESO', 'FECHATRASLADO1', 'SERVICIOTRASLADO1', 'FECHATRASLADO2', 'SERVICIOTRASLADO2', 'FECHATRASLADO3', 'SERVICIOTRASLADO3', 'FECHATRASLADO4', 'SERVICIOTRASLADO4', 'FECHATRASLADO5', 'SERVICIOTRASLADO5', 'FECHATRASLADO6', 'SERVICIOTRASLADO6', 'FECHATRASLADO7', 'SERVICIOTRASLADO7', 'FECHATRASLADO8', 'SERVICIOTRASLADO8', 'FECHATRASLADO9', 'SERVICIOTRASLADO9', 'FECHAALTA', 'SERVICIOALTA', 'TIPOALTA', 'CONDICIONDEALTANEONATO1', 'PESORN1', 'SEXORN1', 'RN1ESTADO', 'CONDICIONDEALTANEONATO2', 'PESORN2', 'SEXORN2', 'RN2ESTADO', 'CONDICIONDEALTANEONATO3', 'PESORN3', 'SEXORN3', 'RN3ESTADO', 'CONDICIONDEALTANEONATO4', 'PESORN4', 'SEXORN4', 'RN4ESTADO', 'DIAGNOSTICO1', 'DIAGNOSTICO2', 'DIAGNOSTICO3', 'DIAGNOSTICO4',

### **2. Comparación de columnas entre años**

#### **Hallazgos principales**

**Datasets con estructura idéntica (mismas columnas)**
- **2019-2020**
- **2019-2022-2023**
- **2020-2022-2023**
- **2022-2023**

**Datasets con estructura distinta**

| Años | Problema | Implicación |
|-----------|----------|------------|
| **2019-2021** | Cambio en nombre de columna: `COD_HOSPITAL` vs `ï»¿COD_HOSPITAL` | Problema de codificación UTF-8 BOM en 2021. La BOM (Byte Order Mark) fue añadida al inicio del archivo. |
| **2020-2021** | Cambio en nombre de columna: `COD_HOSPITAL` vs `ï»¿COD_HOSPITAL` | Mismo problema de codificación BOM. |
| **2021-2022** | Cambio en nombre de columna: `COD_HOSPITAL` vs `ï»¿COD_HOSPITAL` | Inconsistencia en codificación entre 2021 y 2022. |
| **2021-2023** | Cambio en nombre de columna: `COD_HOSPITAL` vs `ï»¿COD_HOSPITAL` | El problema persiste en 2023. |
| **(2019-2023)-2024** | Cambio de identificador: `CIP_ENCRIPTADO` reemplazado por `ID_BENEFICIARIO` | Cambio significativo en 2024: nuevo sistema de identificación de beneficiarios. |

**Conclusiones**

1. **Problema de codificación (2021-2023)**: Los archivos de 2021 contienen un BOM (Byte Order Mark) UTF-8 que causa que la primera columna se nombre `ï»¿COD_HOSPITAL` en lugar de `COD_HOSPITAL`, lo cual requiere limpieza.

2. **Cambio de identificador (2024)**: En 2024 se dejó de usar `CIP_ENCRIPTADO` (código de identidad del paciente encriptado) para usar `ID_BENEFICIARIO`. Esto sugiere un cambio en la política de privacidad o en el sistema de recolección de datos.

3. **Medidas de procedimiento**: Para análisis que combinen datos de 2019-2024, será necesario:
   - Limpiar los nombres de columnas con BOM en 2021-2023
   - Crear una variable de identificación consistente (mapear CIP_ENCRIPTADO a ID_BENEFICIARIO)

In [ ]:
# ==================== COMPARACIÓN DE ESQUEMAS ====================

# Diccionario para almacenar el conjunto de columnas de cada archivo
columnas_por_archivo = {}

# Leer solo los encabezados (sin filas) de cada archivo para eficiencia
for archivo in archivos:
    ruta = os.path.join(carpeta, archivo)
    # nrows=0 carga solo columnas, sin datos
    df = pd.read_csv(ruta, nrows=0)
    # Convertir a conjunto para operaciones de diferencia
    columnas_por_archivo[archivo] = set(df.columns)

# ==================== ANÁLISIS COMPARATIVO ====================

# Comparar todos los pares de archivos posibles (sin repetir comparaciones)
for i in range(len(archivos)):
    for j in range(i+1, len(archivos)):
        a1, a2 = archivos[i], archivos[j]
        cols1, cols2 = columnas_por_archivo[a1], columnas_por_archivo[a2]
        
        # Calcular qué columnas faltan en cada archivo (diferencia de conjuntos)
        faltan_en_a1 = cols2 - cols1  # Columnas en a2 pero no en a1
        faltan_en_a2 = cols1 - cols2  # Columnas en a1 pero no en a2
        
        # Mostrar resultado de comparación
        if faltan_en_a1 or faltan_en_a2:
            # Si hay diferencias, reportarlas
            print(f"ADVERTENCIA: Diferencias entre {a1} y {a2}:")
            if faltan_en_a1:
                print(f"   - {a1} NO tiene: {faltan_en_a1}")
            if faltan_en_a2:
                print(f"   - {a2} NO tiene: {faltan_en_a2}")
        else:
            # Si no hay diferencias, confirmar que son idénticos
            print(f"✔︎✔︎ CORRECTO: {a1} y {a2} tienen las mismas columnas")

✔︎✔︎ CORRECTO: GRD_ONCOLOGIA_2019.csv y GRD_ONCOLOGIA_2020.csv tienen las mismas columnas
ADVERTENCIA: Diferencias entre GRD_ONCOLOGIA_2019.csv y GRD_ONCOLOGIA_2021.csv:
   - GRD_ONCOLOGIA_2019.csv NO tiene: {'ï»¿COD_HOSPITAL'}
   - GRD_ONCOLOGIA_2021.csv NO tiene: {'COD_HOSPITAL'}
✔︎✔︎ CORRECTO: GRD_ONCOLOGIA_2019.csv y GRD_ONCOLOGIA_2022.csv tienen las mismas columnas
✔︎✔︎ CORRECTO: GRD_ONCOLOGIA_2019.csv y GRD_ONCOLOGIA_2023.csv tienen las mismas columnas
ADVERTENCIA: Diferencias entre GRD_ONCOLOGIA_2019.csv y GRD_ONCOLOGIA_2024.csv:
   - GRD_ONCOLOGIA_2019.csv NO tiene: {'ID_BENEFICIARIO'}
   - GRD_ONCOLOGIA_2024.csv NO tiene: {'CIP_ENCRIPTADO'}
ADVERTENCIA: Diferencias entre GRD_ONCOLOGIA_2020.csv y GRD_ONCOLOGIA_2021.csv:
   - GRD_ONCOLOGIA_2020.csv NO tiene: {'ï»¿COD_HOSPITAL'}
   - GRD_ONCOLOGIA_2021.csv NO tiene: {'COD_HOSPITAL'}
✔︎✔︎ CORRECTO: GRD_ONCOLOGIA_2020.csv y GRD_ONCOLOGIA_2022.csv tienen las mismas columnas
✔︎✔︎ CORRECTO: GRD_ONCOLOGIA_2020.csv y GRD_ONCOLOGIA_2023.